# CayleyBeam100H100 Kaggle 2xT4 debug notebook

Purpose: validate dummy backend, CUDA extension build, GPU memory sizing, and 1-2 GPU launch on Kaggle T4. Transformer Engine is intentionally disabled.

In [ ]:
import os, sys, subprocess, textwrap, pathlib, json
import torch

print('python', sys.version)
print('torch', torch.__version__)
print('cuda_available', torch.cuda.is_available())
print('cuda_device_count', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'gpu={i}; name={p.name}; total_memory_GiB={p.total_memory/1024**3:.2f}; capability={p.major}.{p.minor}')

In [ ]:
PROJECT_DIR = pathlib.Path('/kaggle/working/CayleyBeam100H100')
if not PROJECT_DIR.exists():
    candidates = [p for p in pathlib.Path('/kaggle/working').glob('*') if (p / 'beam_engine.py').exists()]
    if candidates:
        PROJECT_DIR = candidates[0]
        
print('PROJECT_DIR', PROJECT_DIR)
print('PROJECT_DIR.exists()', PROJECT_DIR.exists())

# Verify PROJECT_DIR is in /kaggle/working (writable), not /kaggle/input (read-only)
if '/kaggle/input' in str(PROJECT_DIR):
    raise RuntimeError(f'PROJECT_DIR is in read-only /kaggle/input: {PROJECT_DIR}. Copy to /kaggle/working first.')

assert (PROJECT_DIR / 'beam_engine.py').exists(), 'Upload or copy repository to /kaggle/working/CayleyBeam100H100 before running build cells.'
os.chdir(PROJECT_DIR)
print('cwd', pathlib.Path.cwd())
print('Can write to cwd:', os.access(str(PROJECT_DIR), os.W_OK))

In [ ]:
env = {
    'WORLD_SIZE': '2' if torch.cuda.device_count() >= 2 else '1',
    'GLOBAL_BEAM_WIDTH': '4194304',
    'B_MICRO': '32768',
    'SCORE_RING_DEPTH': '16',
    'NET_RING_DEPTH': '2',
    'BUCKET_CAP_PER_PEER': '524288',
    'INFERENCE_BACKEND': 'dummy',
    'USE_CUDA_GRAPHS': '0',
    'NCCL_IB_DISABLE': '1',
    'NCCL_P2P_DISABLE': '0',
    'NCCL_DEBUG': 'WARN',
    'TORCH_CUDA_ARCH_LIST': '7.5',
        'MAX_JOBS': '2',
}
os.environ.update(env)
print(json.dumps(env, indent=2))

In [ ]:
try:
    import transformer_engine
    print('transformer_engine_present=True; Kaggle path still uses dummy backend')
except Exception as exc:
    print('transformer_engine_present=False; expected_for_Kaggle_T4=True; error=', repr(exc))

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_DIR))
import data_loader
import numpy as np

# Load puzzle information
puzzle_info = data_loader.load_puzzle_info()
central_state = data_loader.get_central_state()
generators = data_loader.get_generators()

print("Puzzle Information Loaded:")
print(f"  Central state size: {len(central_state)}")
print(f"  Central state: {central_state}")
print(f"  Number of generators: {len(generators)}")
print(f"  Generator names: {sorted(generators.keys())}")
print()

# Load test puzzles (limit to first 5 for preview)
test_puzzles = data_loader.load_test_puzzles(max_puzzles=5)
print(f"Test Puzzles (first 5):")
for puzzle_id, state in test_puzzles:
    print(f"  ID={puzzle_id}: {state[:10]}... (size={len(state)})")
print()

# Show full test set size
all_test_puzzles = data_loader.load_test_puzzles()
print(f"Total test puzzles available: {len(all_test_puzzles)}")

In [ ]:
subprocess.run([sys.executable, 'scripts/t4_sizing.py'], check=True)

In [ ]:
import shutil
# Ensure build directory is in writable location
build_dir = PROJECT_DIR / 'build'
if build_dir.exists():
    shutil.rmtree(build_dir)
build_dir.mkdir(parents=True, exist_ok=True)

# Build with explicit build directory to avoid Kaggle read-only filesystem issues
subprocess.run([
    sys.executable, 'setup.py', 'build_ext', 
    '--inplace',
    '--build-temp', str(build_dir)
], check=True)

In [ ]:
# Alternative: Use JIT compilation directly from Python (more reliable on Kaggle)
print("Attempting JIT build from beam_engine.build_extension()...")
try:
    from beam_engine import build_extension
    ext = build_extension(verbose=True)
    print("✓ JIT build successful!")
except Exception as e:
    print(f"✗ JIT build failed: {e}")
    print("Falling back to setup.py build_ext...")
    raise

In [ ]:
one_rank_env = os.environ.copy()
one_rank_env.update({'WORLD_SIZE': '1', 'RANK': '0', 'LOCAL_RANK': '0', 'CUDA_VISIBLE_DEVICES': '0'})
subprocess.run([sys.executable, 'beam_engine.py'], env=one_rank_env, check=True)

In [ ]:
if torch.cuda.device_count() >= 2:
    subprocess.run([
        sys.executable, '-m', 'torch.distributed.run',
        '--standalone', '--nnodes=1', '--nproc_per_node=2', 'beam_engine.py'
    ], env=os.environ.copy(), check=True)
else:
    print('skip_2gpu=True; reason=less_than_2_cuda_devices_visible')

Expected success criteria:

- `memory_ok=True` in sizing output.
- Extension build completes.
- Single-rank smoke test prints sizes, counters, threshold.
- Two-rank smoke test completes when two T4 GPUs are visible.